## Eurostat - Data Processing

the idea is: to work further on the variables and to plot.

In [1]:
#import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

print("Everything is working!")

Everything is working!


In [2]:
#import csv file HICP
hicp_merge = pd.read_csv(r"C:\Users\Asus\Documents\DA CareerFoundry\Side Projects\inflacao-dashboard\data\processed\hicp_merged.csv")

In [3]:
#import csv file for house inflation
hpi = pd.read_csv(r"C:\Users\Asus\Documents\DA CareerFoundry\Side Projects\inflacao-dashboard\data\processed\hpi_q_wrangled.csv")

In [4]:
hicp_merge.head(5)

,category,country,date,homologous_inflation,year,item_weight
0,All-items HICP,Germany,2019-01-01,1.7,2019,1000.0
1,All-items HICP,Germany,2019-02-01,1.7,2019,1000.0
2,All-items HICP,Germany,2019-03-01,1.4,2019,1000.0
3,All-items HICP,Germany,2019-04-01,2.1,2019,1000.0
4,All-items HICP,Germany,2019-05-01,1.3,2019,1000.0


In [5]:
hpi.head(5)

,purchase,country,date,house_price_growth_quarterly
0,Purchases of existing dwellings,Germany,2019Q1,0.0
1,Purchases of existing dwellings,Germany,2019Q2,2.8
2,Purchases of existing dwellings,Germany,2019Q3,1.3
3,Purchases of existing dwellings,Germany,2019Q4,2.3
4,Purchases of existing dwellings,Germany,2020Q1,1.1


### Estimating Category Contributions to Inflation

The HICP item weights are expressed in parts per thousand, where the total consumption basket is equal to 1,000. Dividing each item weight by 1,000 converts it into a proportional share of the average household consumption basket.

The estimated contribution to inflation is then calculated by multiplying each category's weight share by its monthly annual rate of change.

$
\text{Estimated contribution (percentage points)} =
\left(\frac{\text{Item weight}}{1000}\right)
\times \text{Annual rate of change}
$

This estimate helps distinguish between categories with large price increases and categories with a larger potential effect on overall consumer inflation because they represent a greater share of the average consumption basket.

The resulting measure should be interpreted with caution. It is an approximation intended for exploratory analysis and does not replace Eurostat’s official contribution-to-inflation calculations. In addition, categories at different ECOICOP levels may overlap. For example, the aggregate category *Food and non-alcoholic beverages* includes more detailed food products such as bread, olive oil and eggs. Therefore, estimated contributions from overlapping categories should not be added together.

In [6]:
hicp_merge["weight_share"] = hicp_merge["item_weight"] / 1000

hicp_merge["estimated_contribution_pp"] = (
    hicp_merge["weight_share"] * hicp_merge["homologous_inflation"]
)

In [7]:
hicp_merge.head(5)

,category,country,date,homologous_inflation,year,item_weight,weight_share,estimated_contribution_pp
0,All-items HICP,Germany,2019-01-01,1.7,2019,1000.0,1.0,1.7
1,All-items HICP,Germany,2019-02-01,1.7,2019,1000.0,1.0,1.7
2,All-items HICP,Germany,2019-03-01,1.4,2019,1000.0,1.0,1.4
3,All-items HICP,Germany,2019-04-01,2.1,2019,1000.0,1.0,2.1
4,All-items HICP,Germany,2019-05-01,1.3,2019,1000.0,1.0,1.3


### Creating a Quarterly Inflation Series

The HICP dataset is published monthly, while the House Price Index (HPI) is published quarterly. To make the two datasets easier to compare, a quarterly HICP inflation series is created.

First, the analysis filters the HICP data to retain only the *All-items HICP* category, which represents overall consumer price inflation. Each monthly observation is then assigned to its corresponding calendar quarter.

The monthly year-on-year inflation rates are grouped by country and quarter, and their arithmetic mean is calculated. The resulting variable, `mean_monthly_yoy_inflation`, represents the average of the three monthly year-on-year HICP inflation rates within each quarter.

This measure is intended to align the time frequency of consumer price inflation with the quarterly HPI dataset. It should not be interpreted as a quarterly inflation rate. Rather, it is a quarterly summary of monthly year-on-year inflation rates.

In [8]:
hicp_merge.dtypes

category                         str
country                          str
date                             str
homologous_inflation         float64
year                           int64
item_weight                  float64
weight_share                 float64
estimated_contribution_pp    float64
dtype: object

In [9]:
#changing datatypes:
hicp_merge["date"] = pd.to_datetime(
    hicp_merge["date"],
    format="%Y-%m-%d",
    errors="raise"
)

In [10]:
#creating a quarterly HICP inflation series:
inflation_quarterly = (
    hicp_merge[hicp_merge["category"].eq("All-items HICP")]
    .assign(quarter=lambda x: x["date"].dt.to_period("Q"))
    .groupby(["country", "quarter"], as_index=False)
    .agg(mean_monthly_yoy_inflation=("homologous_inflation", "mean"))
)

In [11]:
hicp_merge.dtypes

category                                str
country                                 str
date                         datetime64[us]
homologous_inflation                float64
year                                  int64
item_weight                         float64
weight_share                        float64
estimated_contribution_pp           float64
dtype: object

In [12]:
inflation_quarterly

,country,quarter,mean_monthly_yoy_inflation
0,"Euro area (EA11-1999, EA12-2001, EA13-2007, EA...",2019Q1,1.433333
1,"Euro area (EA11-1999, EA12-2001, EA13-2007, EA...",2019Q2,1.400000
2,"Euro area (EA11-1999, EA12-2001, EA13-2007, EA...",2019Q3,0.933333
3,"Euro area (EA11-1999, EA12-2001, EA13-2007, EA...",2019Q4,1.000000
4,"Euro area (EA11-1999, EA12-2001, EA13-2007, EA...",2020Q1,1.100000
...,...,...,...
163,Spain,2024Q4,2.333333
164,Spain,2025Q1,2.666667
165,Spain,2025Q2,2.166667
166,Spain,2025Q3,2.800000


In [13]:
#preparing the quarterly House Price Index dataset:
hpi["date"] = pd.PeriodIndex(hpi["date"], freq="Q")

hpi_total = (
    hpi[hpi["purchase"].eq("Total")]
    .loc[:, ["country", "date", "house_price_growth_quarterly"]]
    .rename(columns={"date": "quarter"})
)

In [14]:
#merging quarterly HICP inflation with total house price growth
inflation_hpi = inflation_quarterly.merge(
    hpi_total,
    on=["country", "quarter"],
    how="left",
    validate="1:1",
    indicator=True
)

assert (inflation_hpi["_merge"] == "both").all()

inflation_hpi = inflation_hpi.drop(columns="_merge")

In [15]:
inflation_hpi.head(5)

,country,quarter,mean_monthly_yoy_inflation,house_price_growth_quarterly
0,"Euro area (EA11-1999, EA12-2001, EA13-2007, EA...",2019Q1,1.433333,0.4
1,"Euro area (EA11-1999, EA12-2001, EA13-2007, EA...",2019Q2,1.400000,1.8
2,"Euro area (EA11-1999, EA12-2001, EA13-2007, EA...",2019Q3,0.933333,1.4
3,"Euro area (EA11-1999, EA12-2001, EA13-2007, EA...",2019Q4,1.000000,0.8
4,"Euro area (EA11-1999, EA12-2001, EA13-2007, EA...",2020Q1,1.100000,1.1


In [16]:
#export merged dataset:
inflation_hpi.to_csv('inflation_hpi.csv', index = False)